In [1]:
import os
import csv
import shutil
import requests
from tqdm import tqdm
from PIL import Image
from io import BytesIO
import pickle
import pandas as pd

import aiohttp
import asyncio

import torch
import numpy as np
from torchvision import transforms
from efficientnet_pytorch import EfficientNet
from xgboost import XGBClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import pairwise_distances
from torchvision import transforms

import shutil
import tempfile

In [20]:
#---------------------------------------
# Chargement des modèles (XGBoost entrainé et EfficientNet) et des données
# ----------------------------------------

model_effnet = EfficientNet.from_pretrained('efficientnet-b3')
model_effnet.eval()

model_xgb = XGBClassifier()
model_xgb.load_model("../backend/app/models_flowers_fruits/xgboost_model.json")

with open("../backend/app/models_flowers_fruits/unique_classes.txt", "r") as f:
    unique_classes = f.read().splitlines()
    print("Unique classes loaded successfully.")

list_plants2 = pd.read_csv("../data/list_plants3.csv")
list_plants = list(list_plants2["name"].unique())
sorted_plants = sorted(list_plants)

Loaded pretrained weights for efficientnet-b3
Unique classes loaded successfully.


In [5]:
#---------------------------------
# liste des plantes utilisées pour l'entraînement du modèle de classification de plantes
# cette liste est utilisée pour faire le mapping entre les classes prédites par le modèle et les noms communs des plantes
#----------------------------------
    

plants_list = {
    "apple":            "Malus domestica",
    "blackberry":        "Rubus fruticosus",
    "orange":           "Citrus sinensis",
    "grape":            "Vitis vinifera",
    "strawberry":       "Fragaria × ananassa",
    "raspberry":        "Rubus idaeus",
    "watermelon":       "Citrullus lanatus",
    "pineapple":        "Ananas comosus",
    "mango":            "Mangifera indica",
    "blueberry":        "Vaccinium angustifolium",
    "peach":            "Prunus persica",
    "kiwi":             "Actinidia deliciosa",
    "pear":             "Pyrus calleryana",
    "plum":             "Prunus domestica",
    "cherry":           "Prunus avium",
    "grapefruit":       "Citrus × paradisi",
    "melon":            "Cucumis melo",
    "papaya":           "Carica papaya",
    "fig":              "Ficus carica",
    "pomegranate":       "Punica granatum",
    "avocado":          "Persea americana",
    "coconut":          "Cocos nucifera",
    "banana":           "Musa",
    "lime":             "Citrus aurantiifolia",
    "lemon":            "Citrus limon",
    "raspberry":        "Rubus idaeus",
    "blackberry":        "Rubus fruticosus",
    "cranberry":        "Vaccinium macrocarpon",
    "passionfruit":      "Passiflora edulis",
    "guava":            "Psidium guajava",
    "lychee":           "Litchi chinensis",
    "sunflower":        "Helianthus annuus",
    "daisy":            "Bellis perennis",
    "lily":             "Lilium",
    "peony":            "Paeonia",
    "dahlia":           "Dahlia",
    "chrysanthemum":    "Chrysanthemums",
    "iris":             "Iris",
    "poppy":            "Papaver rhoeas",
    "hydrangea":        "Hydrangea",
    "carnation":        "Dianthus caryophyllus",
    "freesia":          "Freesia",
    "gerbera":          "Gerbera jamesonii",
    "anemone":          "Anemone",
    "ranunculus":       "Ranunculus",
    "protea":           "Protea",
    "bird_of_paradise": "Strelitzia reginae",
    "lisianthus":       "Eustoma grandiflorum",
    "gypsophila":       "Gypsophila paniculata",
    "hellebore":        "Helleborus",
    "anthurium":        "Anthurium",
    "allium":           "Allium",
    "cosmos":           "Cosmos bipinnatus",
    "zinnia":           "Zinnia elegans",
    "foxglove":         "Digitalis purpurea",
    "lavender":         "Lavandula",
    "wisteria":         "Wisteria",
    "basil":            "Ocimum basilicum",
    "lemon_verbena":    "Aloysia citrodora",
    "coriander":        "Coriandrum sativum",
    "thyme":            "Thymus vulgaris",
    "chives":           "Allium schoenoprasum",
    "savory":           "Satureja hortensis",
    "dill":             "Anethum graveolens",
    "coriander":        "Coriandrum sativum",
    "angelica":         "Angelica archangelica",
    "oregano":          "Origanum vulgare",
    "fennel":           "Foeniculum vulgare",  
    "tarragon":         "Artemisia dracunculus",
    "parsley":          "Petroselinum crispum",
    "mint":             "Mentha",
    "chamomile":        "Matricaria chamomilla",
    "lovage":           "Levisticum officinale",
    "lavender":         "Lavandula",
    "mugwort":          "Artemisia vulgaris",
    "rosemary":         "Salvia rosmarinus",
    "sage":             "Salvia officinalis",
    "lemongrass":       "Cymbopogon citratus",
    "bay_leaf":         "Laurus nobilis",
    "marjoram":         "Origanum majorana",
    "savory":           "Satureja hortensis",
    "anise":            "Pimpinella anisum",
    "stevia":           "Stevia rebaudiana",
    "lemon_balm":       "Melissa officinalis",
    "pandan":           "Pandanus amaryllifolius",
    "holy_basil":       "Ocimum tenuiflorum",
    "lemon_verbena":    "Aloysia citrodora",
    "wintergreen":      "Gaultheria procumbens",
    "hyssop":           "Hyssopus officinalis",   
    "borage":           "Borago officinalis"
}

In [30]:
# ---------------------------
# Définition des fonctions de téléchargement + redimensionnement
# une fonction asynchrone a été utilisée pour le téléchargement de plusieurs images en parallèle, ce qui accélère considérablement le processus de collecte d'images à partir d'URLs
# ---------------------------

def download_and_resize(url, filepath):

    try:
        r = requests.get(url, timeout=15)

        img = Image.open(BytesIO(r.content)).convert("RGB")

        img = img.resize(IMAGE_SIZE)

        img.save(filepath, "JPEG", quality=90)

        return True

    except:
        return False

async def download_and_resize_async(session, url, filepath, timeout=10):
    """Download and resize a single image asynchronously."""
    try:
        async with session.get(url, timeout=aiohttp.ClientTimeout(total=timeout)) as resp:
            if resp.status == 200:
                img = Image.open(BytesIO(await resp.read())).convert("RGB")
                img = img.resize(IMAGE_SIZE)
                img.save(filepath, "JPEG", quality=90)
                return True
    except Exception as e:
        print(f"    [ERROR] {str(e)[:60]}", end="")
    return False

async def download_batch_async(urls_filepaths, max_concurrent=10):
    """
    Download multiple images concurrently.
    
    Args:
        urls_filepaths: list of (url, filepath) tuples
        max_concurrent: max parallel downloads (10–20 is good; don't exceed ~50)
    
    Returns:
        count of successful downloads
    """
    success = 0
    semaphore = asyncio.Semaphore(max_concurrent)
    
    async def bounded_download(session, url, filepath):
        nonlocal success
        async with semaphore:
            if await download_and_resize_async(session, url, filepath):
                success += 1
            return success
    
    connector = aiohttp.TCPConnector(limit_per_host=5)
    async with aiohttp.ClientSession(connector=connector) as session:
        tasks = [bounded_download(session, url, fp) for url, fp in urls_filepaths]
        await asyncio.gather(*tasks)
    
    return success

In [39]:
#---------------------------
# Fonction de filtrage d'une image basée sur la prédiction du modèle XGBoost et la distance au centroïde de la classe prédite
#---------------------------

def filter_image(embedding, clusters, model_xgb, threshold_factor=1.2):
    # 1. Prédiction XGBoost
    proba = model_xgb.predict_proba([embedding])[0]
    pred_class = unique_classes[np.argmax(proba)]
    confidence = np.max(proba)

    # 2. Distance au centroïde
    centroid = clusters[pred_class]["centroid"]
    dist = np.linalg.norm(embedding - centroid)

    # 3. Seuil dynamique
    threshold = clusters[pred_class]["threshold"] * threshold_factor

    # 4. Décision
    is_valid = (confidence > 0.1) and (dist < threshold)

    return {
        "pred_class": pred_class,
        "confidence": confidence,
        "distance": dist,
        "threshold": threshold,
        "is_valid": is_valid
    }

In [40]:
#---------------------------
# Fonction principale pour vérifier si une image est valide ou non en utilisant le modèle EfficientNet pour extraire les caractéristiques, 
# puis en utilisant le modèle XGBoost pour faire la prédiction et le filtrage basé sur la distance au centroïde de la classe prédite
#---------------------------

def is_image_valid(img_path, model_effnet, scaler, clusters, model_xgb, threshold_factor=1.2):
    try:
        # Charger l'image
        img = Image.open(img_path).convert("RGB")
        x = transform(img).unsqueeze(0)

        # Embedding EfficientNet
        with torch.no_grad():
            feat = model_effnet.extract_features(x)
            feat = torch.nn.functional.adaptive_avg_pool2d(feat, 1).flatten().numpy()
            
        # Normalisation
        feat_scaled = scaler.transform([feat])[0]
        # Filtrage
        result = filter_image(
            feat_scaled,
            clusters,
            model_xgb,
            threshold_factor=threshold_factor
        )
        return result["is_valid"]

    except Exception as e:
        print(f"Erreur lors du filtrage de {img_path}: {e}")
        return False

In [41]:
# ----------------------------
# Les embeding des 8000 premières images de plantes utilisées pour l'entraînement du modèle XGBoost
# Extraite sur Google Colab et ensuite sauvegarder en fichier .npy pour être utilisé dans le processus de filtrage des images téléchargées à partir d'URLs
# ----------------------------

embeddings = np.load("../data/embeddings.npy") 
labels = np.load("../data/labels.npy")      

scaler = StandardScaler()
embeddings_scaled = scaler.fit_transform(embeddings)     

print("Embeddings shape:", embeddings_scaled.shape)
print("Labels shape:", labels.shape)

Embeddings shape: (8001, 1536)
Labels shape: (8001,)


In [42]:
#------------------------------------
# Génération des clusters pour chaque classe à partir des embeddings, en utilisant KMeans pour modéliser la distribution interne de chaque classe et calculer les distances au centroïde
#------------------------------------

unique_classes = np.unique(labels)
clusters = {}

for cls in unique_classes:
    idx = np.where(labels == cls)[0]
    X_cls = embeddings_scaled[idx]

    # KMeans avec 4 cluster (centroïde principal)
    kmeans = KMeans(n_clusters=4, random_state=42)
    kmeans.fit(X_cls)

    centroid = kmeans.cluster_centers_[0]
    distances = pairwise_distances(X_cls, [centroid]).flatten()

    clusters[cls] = {
        "centroid": centroid,
        "distances": distances,
        "threshold": np.percentile(distances, 99) * 1.2,  # seuil automatique
        "indices": idx
    }

#    print(f"{cls}: threshold={clusters[cls]['threshold']:.4f}")

In [43]:
# ---------------------------
# PARAMETRES
# ---------------------------

DATASET_DIR       = "../data/dataset_test2"
IMAGES_PER_FLOWER = 40   # images to download per flower
IMAGE_SIZE        = (400, 400)
BATCH_SIZE        = 20    # images to download per round

# Transformations
transform = transforms.Compose([
    transforms.Resize((300, 300)),
    transforms.ToTensor(),
])


# ---------------------------
# CREATION DOSSIERS
# ---------------------------

os.makedirs(DATASET_DIR, exist_ok=True)
csv_rows = []

for plant in sorted_plants:
    if plant in plants_list.keys():
        print(f"\n{'='*60}")
        print(f"  {plant}")
        fruit_name = plant
        taxon = plants_list[fruit_name]
        #IMAGES_PER_FLOWER = plants.remaining

    fruit_dir = os.path.join(DATASET_DIR, fruit_name)
    #os.makedirs(fruit_dir, exist_ok=True)
    
    tested_urls = set()   

    saved_count   = 0   # images saved so far
    saved_count2  = 0   # images rejected so far
    url_idx       = 0   # pointer into all_urls
    page          = 1
    all_urls      = []
    all_dates     = []
    no_more_pages = False

    pbar = tqdm(total=IMAGES_PER_FLOWER, desc=f"  Saving {fruit_name}")

    while saved_count < IMAGES_PER_FLOWER:

        # ── 1. Fetch more URLs from the API until we have a full batch ──────
        while len(all_urls) - url_idx < BATCH_SIZE and not no_more_pages:
            api_url = "https://api.inaturalist.org/v1/observations"
            params  = {
                "taxon_name":    taxon,
                "quality_grade": "casual,research",
                "identified":    "true",
                "photos":        "true",
                "per_page":      200,
                "page":          page,
                "month":         "5,6,7,8,9",
                "year":          "2024, 2025",
                "order":         "asc",
            }
            results = requests.get(api_url, params=params).json().get("results", [])
            if not results:
                no_more_pages = True
                break
            for obs in results:
                for photo in obs["photos"]:
                    all_urls.append(photo["url"].replace("square", "large"))
                    all_dates.append(obs.get("observed_on", ""))
            page += 1

        # ── 2. Build the next batch ─────────────────────────────────────────
        batch_end   = min(url_idx + BATCH_SIZE, len(all_urls))
        batch_urls  = all_urls[url_idx:batch_end]
        batch_dates = all_dates[url_idx:batch_end]

        if not batch_urls:
            print(f"  No more URLs available. Saved {saved_count}/{IMAGES_PER_FLOWER}")
            break

        # fichiers temporaires
        temp_paths = [
            os.path.join(DATASET_DIR, f"temp_{i}.jpg")
            for i in range(500, 1500, 1)
        ]

        urls_filepaths = list(zip(batch_urls, temp_paths))


        # ── 3. Async download of the batch ──────────────────────────────────
        await download_batch_async(urls_filepaths, max_concurrent=10)

        # 4. Record saved images
        for temp_path, date_val in zip(temp_paths, batch_dates):
            url_idx += 1
            
            if not os.path.exists(temp_path):
                print(f"  Failed to download: {temp_path}")
                continue

            # --- Filtrage automatique ---
            is_valid = is_image_valid(
                temp_path,
                model_effnet,
                scaler,
                clusters,
                model_xgb,
                threshold_factor=1.2
            )
            print(is_valid)
            if not is_valid:
                final_path = os.path.join(DATASET_DIR, "rejected",
                f"{fruit_name}_{saved_count2+ 30 :05d}.jpg"
            )
                os.rename(temp_path, final_path)
                saved_count2 += 1
                continue

            # --- Si valide : enregistrer dans le CSV ---
            final_path = os.path.join(DATASET_DIR, "accepted",
                f"{fruit_name}_{saved_count+ 30 :05d}.jpg")
            os.rename(temp_path, final_path)

            csv_rows.append([
                fruit_name,
                os.path.join(fruit_name, os.path.basename(final_path)),
                date_val,
            ])

            saved_count += 1
            pbar.update(1)

            if saved_count >= IMAGES_PER_FLOWER:
                break
            if no_more_pages and url_idx >= len(all_urls):
                print(f"  Exhausted all URLs. Saved {saved_count}/{IMAGES_PER_FLOWER}")
                break


    pbar.close()
    print(f"  ✓ {saved_count}/{IMAGES_PER_FLOWER} images saved for {fruit_name} for a total of {len(all_urls)} URLs tested.")

CSV_FILE = os.path.join(DATASET_DIR, "dataset_test.csv")
with open(CSV_FILE, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["fruit_type", "image_path", "date_observed"])
    writer.writerows(csv_rows)

print("\nDataset et CSV créés avec succès.")


  allium


  Saving allium:   2%|▎         | 1/40 [00:03<02:26,  3.75s/it]

True
False


  Saving allium:   5%|▌         | 2/40 [00:04<01:14,  1.96s/it]

True


...

  Saving allium:  48%|████▊     | 19/40 [00:17<00:12,  1.63it/s]

True


  Saving allium:  50%|█████     | 20/40 [00:17<00:10,  1.86it/s]

True
False
False
False
False
False
False


CancelledError: 